# Fine-tuning Qwen3-VL-4B with LoRA

In this notebook we teach a vision-language model to answer questions about document pages **and cite the page it used**.

We will:

1. load a small dataset of questions about document pages,
2. turn each question into a chat example the model can learn from,
3. add LoRA adapters to the model,
4. train with TRL's `SFTTrainer`,
5. try the fine-tuned model.

Training runs on one RTX 4090 in about 13 minutes.

## Loading the dataset

We use the **Computer Science** part of ViDoRe V3. It has the same format as our pharmaceuticals data, but it is a
different domain, so our dev and test questions stay untouched.

The dataset has three parts: `corpus` (the pages), `queries` (the questions) and `qrels` (which pages answer which question).

In [ ]:
from datasets import load_dataset

repo = "vidore/vidore_v3_computer_science"
pages = load_dataset(repo, "corpus")["test"]
questions = load_dataset(repo, "queries")["test"]
qrels = load_dataset(repo, "qrels")["test"]

questions = questions.filter(lambda lang: lang == "english", input_columns="language")
print(pages)
print(questions)

Let's look at one question and its reference answer:

In [ ]:
q = questions[0]
print("Question:", q["query"])
print("Answer:  ", q["answer"])

## Finding the right pages

For every question, `qrels` lists the pages that contain the answer. Let's collect them in a dictionary.

In [ ]:
from collections import defaultdict

correct_pages = defaultdict(list)          # query_id -> list of corpus_ids
for qid, cid in zip(qrels["query_id"], qrels["corpus_id"]):
    correct_pages[qid].append(cid)

print("Correct pages for the first question:", correct_pages[q["query_id"]])

If we only ever show the model correct pages, it learns to cite whatever it sees. So we also add **distractor
pages**: pages that *look* relevant (found by keyword search with BM25) but are not. The model has to pick.

In [ ]:
import bm25s

search = bm25s.BM25()
search.index(bm25s.tokenize(pages["markdown"], stopwords="english"))
results, _ = search.retrieve(bm25s.tokenize(questions["query"], stopwords="english"), k=20)

page_ids = pages["corpus_id"]               # position in the corpus -> page id

def distractors(i, qid, n):
    """The top keyword-search hits for question i that are not correct pages."""
    hits = [page_ids[p] for p in results[i]]
    return [c for c in hits if c not in correct_pages[qid]][:n]

print("Distractors for the first question:", distractors(0, q["query_id"], 2))

## Building the training examples

Each example is a short chat:

- **system**: the instructions,
- **user**: 3 page images and the question,
- **assistant**: the answer we want, as JSON with the page ids it should cite.

The model only learns from the assistant part.

In [ ]:
import json
import random

SYSTEM = (
    "You answer questions about documents using only the pages you are given. "
    "Answer in one or two sentences. "
    "In citations, list the ids of the pages that contain the answer. "
    "If the pages do not contain the answer, set abstain to true and leave answer empty. "
    'Reply with JSON only: {"answer": string, "citations": [page ids], "abstain": boolean}.'
)
row_of = {cid: i for i, cid in enumerate(page_ids)}

def page_image(cid):
    img = pages[row_of[cid]]["image"].convert("RGB")
    img.thumbnail((1280, 1280))            # keep images small: fewer tokens, less memory
    return img

random.seed(0)
examples = []
for i, q in enumerate(questions):
    right = correct_pages[q["query_id"]][:2]
    shown = right + distractors(i, q["query_id"], 3 - len(right))
    random.shuffle(shown)

    page_list = ", ".join(f"Page {c}" for c in shown)
    target = {"answer": q["answer"], "citations": [c for c in shown if c in right], "abstain": False}
    examples.append({
        "images": [page_image(c) for c in shown],
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"The pages above are, in order: {page_list}.\nQuestion: {q['query']}"},
        ],
        "completion": [{"role": "assistant", "content": json.dumps(target)}],
    })

print(len(examples), "examples")

Here is one example, exactly as the model will see it:

In [ ]:
ex = examples[0]
print("USER:", ex["prompt"][1]["content"])
print("ASSISTANT:", ex["completion"][0]["content"])
ex["images"][0]

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(examples)
train_dataset

## Loading the model

`AutoProcessor` turns images and text into model inputs; `AutoModelForImageTextToText` is the model itself.
We load the weights in `bfloat16` to halve the memory.

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "Qwen/Qwen3-VL-4B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=torch.bfloat16)

## Adding LoRA

Instead of updating all 4.4 billion weights, LoRA adds small trainable matrices next to some layers and keeps
everything else frozen. We add them to the attention and MLP layers of the language model.

- `r=16`: the size of the small matrices,
- `lora_alpha=32`: how strongly they count.

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

## Training

`SFTConfig` holds the training settings. The ones worth knowing:

- `per_device_train_batch_size=1` with `gradient_accumulation_steps=8`: the weights update after every 8 examples,
- `num_train_epochs=2`: every example is seen twice,
- `gradient_checkpointing=True`: saves memory by recomputing activations,
- `max_length=None`: never cut an example short (it would lose its answer).

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="../checkpoints/toy-sft-cs",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=3,
    bf16=True,
    gradient_checkpointing=True,
    max_length=None,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Only about 0.7% of the weights are trained. Now we train; the loss is printed at every step:

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("../checkpoints/toy-sft-cs")      # saves only the small LoRA adapter

## Trying the fine-tuned model

Let's ask the trained model the first question again and see what it writes.

In [ ]:
ex = examples[0]
messages = [
    ex["prompt"][0],
    {"role": "user", "content": [*[{"type": "image"} for _ in ex["images"]],
                                 {"type": "text", "text": ex["prompt"][1]["content"]}]},
]
text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = processor(text=[text], images=ex["images"], return_tensors="pt").to(model.device)

with torch.no_grad():
    output = trainer.model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## How well did it do?

Training loss only tells us the model fits these 215 examples. The real test is the pharmaceuticals dev set,
answered by the base model and by the fine-tuned model, both graded by the same 8B judge.
The overnight run (`scripts/toy/overnight.sh`) does exactly that; here is its table:

In [ ]:
from pathlib import Path

latest = sorted(Path("../data/runs").glob("toy-*"))[-1]
print((latest / "summary.txt").read_text())